<a href="https://colab.research.google.com/github/neelay8975/GenAi_prac1/blob/main/GenAI_Prac4_48.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.20.0


In [12]:
# Small demonstration dataset
# Format: English, Hindi, Marathi

data = [
    ["hello", "नमस्ते", "नमस्कार"],
    ["how are you", "आप कैसे हैं", "तुम्ही कसे आहात"],
    ["i am fine", "मैं ठीक हूँ", "मी ठीक आहे"],
    ["what is your name", "आपका नाम क्या है", "तुमचे नाव काय आहे"],
    ["my name is rahul", "मेरा नाम राहुल है", "माझे नाव राहुल आहे"],
    ["where are you going", "आप कहाँ जा रहे हैं", "तुम्ही कुठे जात आहात"],
    ["i am going home", "मैं घर जा रहा हूँ", "मी घरी जात आहे"],
    ["good morning", "सुप्रभात", "शुभ प्रभात"],
    ["good night", "शुभ रात्रि", "शुभ रात्री"],
    ["thank you", "धन्यवाद", "धन्यवाद"],
    ["please help me", "कृपया मेरी मदद करें", "कृपया मला मदत करा"],
    ["what are you doing", "आप क्या कर रहे हैं", "तुम्ही काय करत आहात"],
    ["i am studying", "मैं पढ़ाई कर रहा हूँ", "मी अभ्यास करत आहे"],
    ["i like music", "मुझे संगीत पसंद है", "मला संगीत आवडते"],
    ["i love india", "मुझे भारत से प्यार है", "मला भारत आवडतो"],
    ["this is my house", "यह मेरा घर है", "हे माझे घर आहे"],
    ["open the door", "दरवाजा खोलो", "दरवाजा उघडा"],
    ["close the window", "खिड़की बंद करो", "खिडकी बंद करा"],
    ["come here", "यहाँ आओ", "इथे या"],
    ["sit down", "बैठ जाओ", "बसा"],
    ["stand up", "खड़े हो जाओ", "उभे रहा"],
    ["i am hungry", "मुझे भूख लगी है", "मला भूक लागली आहे"],
    ["i am thirsty", "मुझे प्यास लगी है", "मला तहान लागली आहे"],
    ["what time is it", "अभी कितने बजे हैं", "आत्ता किती वाजले आहेत"],
    ["see you tomorrow", "कल मिलते हैं", "उद्या भेटू"],
    ["have a nice day", "आपका दिन शुभ हो", "तुमचा दिवस चांगला जावो"],
    ["i understand", "मैं समझता हूँ", "मला समजते"],
    ["i do not understand", "मैं नहीं समझता", "मला समजत नाही"],
    ["where do you live", "आप कहाँ रहते हैं", "तुम्ही कुठे राहता"],
    ["i live in nagpur", "मैं नागपुर में रहता हूँ", "मी नागपूरमध्ये राहतो"]
]

In [13]:
# Hum model ko 6 translation directions par train karenge:
# English -> Hindi
# Hindi -> English
# Hindi -> Marathi
# Marathi -> Hindi
# English -> Marathi
# Marathi -> English

pairs = []

for eng, hin, mar in data:

    pairs.append((eng, hin))
    pairs.append((hin, eng))

    pairs.append((hin, mar))
    pairs.append((mar, hin))

    pairs.append((eng, mar))
    pairs.append((mar, eng))

print("Total training pairs:", len(pairs))

Total training pairs: 180


In [14]:
SOURCE_TEXTS = [p[0] for p in pairs]
TARGET_TEXTS = ["<start> " + p[1] + " <end>" for p in pairs]

source_tokenizer = Tokenizer(filters='')
target_tokenizer = Tokenizer(filters='')

source_tokenizer.fit_on_texts(SOURCE_TEXTS)
target_tokenizer.fit_on_texts(TARGET_TEXTS)

source_sequences = source_tokenizer.texts_to_sequences(SOURCE_TEXTS)
target_sequences = target_tokenizer.texts_to_sequences(TARGET_TEXTS)

max_source_len = max(len(x) for x in source_sequences)
max_target_len = max(len(x) for x in target_sequences)

source_sequences = pad_sequences(
    source_sequences,
    maxlen=max_source_len,
    padding='post'
)

target_sequences = pad_sequences(
    target_sequences,
    maxlen=max_target_len,
    padding='post'
)

source_vocab_size = len(source_tokenizer.word_index) + 1
target_vocab_size = len(target_tokenizer.word_index) + 1

print("Source vocabulary:", source_vocab_size)
print("Target vocabulary:", target_vocab_size)
print("Max source length:", max_source_len)
print("Max target length:", max_target_len)

Source vocabulary: 168
Target vocabulary: 170
Max source length: 5
Max target length: 7


In [15]:
latent_dim = 256
embedding_dim = 128

# ---------------- ENCODER ----------------

encoder_inputs = Input(
    shape=(None,),
    name="encoder_input"
)

encoder_embedding = Embedding(
    source_vocab_size,
    embedding_dim,
    mask_zero=True,
    name="encoder_embedding"
)(encoder_inputs)

encoder_lstm = LSTM(
    latent_dim,
    return_state=True,
    name="encoder_lstm"
)

encoder_outputs, state_h, state_c = encoder_lstm(
    encoder_embedding
)

encoder_states = [state_h, state_c]


# ---------------- DECODER ----------------

decoder_inputs = Input(
    shape=(None,),
    name="decoder_input"
)

decoder_embedding_layer = Embedding(
    target_vocab_size,
    embedding_dim,
    mask_zero=True,
    name="decoder_embedding"
)

decoder_embedding = decoder_embedding_layer(
    decoder_inputs
)

decoder_lstm = LSTM(
    latent_dim,
    return_sequences=True,
    return_state=True,
    name="decoder_lstm"
)

decoder_outputs, _, _ = decoder_lstm(
    decoder_embedding,
    initial_state=encoder_states
)

decoder_dense = Dense(
    target_vocab_size,
    activation="softmax",
    name="decoder_output"
)

decoder_outputs = decoder_dense(decoder_outputs)


# Complete model
model = Model(
    [encoder_inputs, decoder_inputs],
    decoder_outputs
)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_input       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_input       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_embedding   │ (None, None, 128) │     21,504 │ encoder_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, None)      │          0 │ encoder_input[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, None, 128) │     21,760 │ decoder_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 256),     │    394,240 │ encoder_embeddin… │
│                     │ (None, 256),      │            │ not_equal[0][0]   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, None,     │    394,240 │ decoder_embeddin… │
│                     │ 256), (None,      │            │ encoder_lstm[0][… │
│                     │ 256), (None,      │            │ encoder_lstm[0][… │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_output      │ (None, None, 170) │     43,690 │ decoder_lstm[0][… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 875,434 (3.34 MB)

 Trainable params: 875,434 (3.34 MB)

 Non-trainable params: 0 (0.00 B)

In [16]:
# Decoder input:
# <start> I am fine <end>
#
# Decoder output:
# I am fine <end>

decoder_input_data = target_sequences[:, :-1]

decoder_target_data = target_sequences[:, 1:]

# Sparse categorical cross entropy ke liye last dimension add
decoder_target_data = np.expand_dims(
    decoder_target_data,
    -1
)

print("Encoder input:", source_sequences.shape)
print("Decoder input:", decoder_input_data.shape)
print("Decoder target:", decoder_target_data.shape)

Encoder input: (180, 5)
Decoder input: (180, 6)
Decoder target: (180, 6, 1)


In [17]:
history = model.fit(
    [source_sequences, decoder_input_data],
    decoder_target_data,
    batch_size=16,
    epochs=200,
    validation_split=0.1,
    verbose=1
)

Epoch 1/200
11/11 ━━━━━━━━━━━━━━━━━━━━ 6s 110ms/step - accuracy: 0.2617 - loss: 5.0614 - val_accuracy: 0.1800 - val_loss: 4.8026
Epoch 2/200
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 52ms/step - accuracy: 0.2518 - loss: 4.0790 - val_accuracy: 0.2400 - val_loss: 4.4045
Epoch 3/200
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 50ms/step - accuracy: 0.3563 - loss: 3.5749 - val_accuracy: 0.3400 - val_loss: 4.0464
Epoch 4/200
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 53ms/step - accuracy: 0.3857 - loss: 3.3572 - val_accuracy: 0.3400 - val_loss: 4.1987
Epoch 5/200
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 51ms/step - accuracy: 0.3857 - loss: 3.2034 - val_accuracy: 0.3400 - val_loss: 4.1407
Epoch 6/200
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 53ms/step - accuracy: 0.3968 - loss: 3.1100 - val_accuracy: 0.3700 - val_loss: 4.2615
Epoch 7/200
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 52ms/step - accuracy: 0.4029 - loss: 3.0326 - val_accuracy: 0.3700 - val_loss: 4.4855
Epoch 8/200
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 49ms/step - accuracy: 0.4042 - loss: 2.9649 - val_accuracy: 0

In [18]:
# Encoder inference model

encoder_model = Model(
    encoder_inputs,
    encoder_states
)

# Decoder inference inputs
decoder_state_input_h = Input(
    shape=(latent_dim,),
    name="decoder_h"
)

decoder_state_input_c = Input(
    shape=(latent_dim,),
    name="decoder_c"
)

decoder_states_inputs = [
    decoder_state_input_h,
    decoder_state_input_c
]

decoder_single_embedding = decoder_embedding_layer(
    decoder_inputs
)

decoder_outputs_inf, state_h_inf, state_c_inf = decoder_lstm(
    decoder_single_embedding,
    initial_state=decoder_states_inputs
)

decoder_states_inf = [
    state_h_inf,
    state_c_inf
]

decoder_outputs_inf = decoder_dense(
    decoder_outputs_inf
)

decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs_inf] + decoder_states_inf
)

print("Inference models ready!")

Inference models ready!


In [19]:
reverse_target_word_index = {
    value: key
    for key, value in target_tokenizer.word_index.items()
}

def translate_sentence(sentence):

    # Input tokenize
    sequence = source_tokenizer.texts_to_sequences([sentence.lower()])

    sequence = pad_sequences(
        sequence,
        maxlen=max_source_len,
        padding='post'
    )

    # Encoder
    states_value = encoder_model.predict(
        sequence,
        verbose=0
    )

    # <start>
    start_token = target_tokenizer.word_index["<start>"]

    end_token = target_tokenizer.word_index["<end>"]

    target_seq = np.array([[start_token]])

    decoded_sentence = []

    for _ in range(max_target_len):

        output_tokens, h, c = decoder_model.predict(
            [target_seq] + states_value,
            verbose=0
        )

        sampled_token_index = np.argmax(
            output_tokens[0, -1, :]
        )

        sampled_word = reverse_target_word_index.get(
            sampled_token_index,
            ""
        )

        if sampled_token_index == end_token:
            break

        if sampled_word != "<start>":
            decoded_sentence.append(sampled_word)

        target_seq = np.array([
            [sampled_token_index]
        ])

        states_value = [h, c]

    return " ".join(decoded_sentence)

In [22]:
language_map = {
    "english": "english",
    "eng": "english",
    "hindi": "hindi",
    "hin": "hindi",
    "marathi": "marathi",
    "mar": "marathi"
}

print("Available languages:")
print("1. English")
print("2. Hindi")
print("3. Marathi")

source_language = input(
    "\nEnter source language: "
).strip().lower()

target_language = input(
    "Enter target language: "
).strip().lower()

source_language = language_map.get(
    source_language,
    source_language
)

target_language = language_map.get(
    target_language,
    target_language
)

if source_language not in ["english", "hindi", "marathi"]:
    print("Invalid source language!")

elif target_language not in ["english", "hindi", "marathi"]:
    print("Invalid target language!")

elif source_language == target_language:
    print("Source aur target language same nahi honi chahiye.")

else:
    sentence = input(
        "\nEnter your sentence: "
    )

    result = translate_sentence(sentence)

    print("\n-----------------------------")
    print("Source :", sentence)
    print("From   :", source_language)
    print("To     :", target_language)
    print("Result :", result)
    print("-----------------------------")

Available languages:
1. English
2. Hindi
3. Marathi

Enter source language: English
Enter target language: Hindi

Enter your sentence: how are you baby

-----------------------------
Source : how are you baby
From   : english
To     : hindi
Result : आप कैसे हैं
-----------------------------
